<a href="https://colab.research.google.com/github/JorgeMiceli1967/IA-SCRAPPING/blob/main/RELEVAMIENTO_IA_MPF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
# ============================================================
# SISTEMA DE RELEVAMIENTO WEB TEMÁTICO PARA ESTADO DEL ARTE
# ------------------------------------------------------------
# Versión: Colab / Python standalone
# Motor de búsqueda: DDGS (DuckDuckGo Search API no oficial)
#
# DESCRIPCIÓN GENERAL
# ------------------------------------------------------------
# Este script implementa un pipeline automatizado para la
# construcción de corpus documental orientado a revisiones
# sistemáticas y estados del arte en dominios interdisciplinarios.
#
# Permite generar consultas de búsqueda a partir de:
#   un producto cartesiano completo entre listas de keywords
#   (por ejemplo, dominio jurídico × dominio informático)
#
#
#
# Las consultas se ejecutan de manera:
#   - libre (búsqueda abierta)
#   - restringida a repositorios específicos mediante operadores site:
#
# FUNCIONALIDADES PRINCIPALES
# ------------------------------------------------------------
# - Generación sistemática y configurable de queries
# - Integración de búsqueda libre y búsqueda en fuentes priorizadas
# - Extracción de resultados (título, snippet, URL)
# - Normalización y estructuración de datos
# - Clasificación heurística del tipo de actor (academia, Estado, empresa, etc.)
# - Desduplicación de URLs (estrategia exacta o canónica)
# - Filtrado de dominios excluidos
# - Manejo automático de acentos
# - Monitoreo de progreso en tiempo real (% de avance)
# - Exportación a CSV y Excel
#
# VARIABLES CONFIGURABLES CLAVE
# ------------------------------------------------------------
# - Producto cartesiano entre listas de keywords
# - Cantidad de resultados por consulta
# - Límite de queries generadas
# - Repositorios forzados (site:)
# - Inclusión o no de búsqueda libre
# - Activación y estrategia de desduplicación
# - Frecuencia de consultas (delay entre requests)
# - Lista de dominios excluidos
#
# SALIDA
# ------------------------------------------------------------
# Dataset estructurado con:
#   CAMPO, PAIS, DOMINIO, TITULO, RESUMEN, URL, ACTOR, CONSULTA
#
# Uso típico:
#   - construcción de estados del arte
#   - relevamientos exploratorios
#   - identificación de actores institucionales
#   - análisis preliminar de literatura y fuentes
#
# NOTA
# ------------------------------------------------------------
# El script está diseñado para ser auto-contenido, reproducible
# y fácilmente extensible (nuevos dominios, nuevos repositorios,
# nuevas heurísticas de clasificación o filtrado).
# ============================================================

import os
os.environ["PYTHONWARNINGS"] = "ignore"

import warnings
warnings.simplefilter("ignore")

import sys
import subprocess
import re
import time
from urllib.parse import urlparse, urlunparse

# ---------- instalar paquetes ----------
def ensure_package(package_name: str):
    try:
        __import__(package_name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", package_name])

ensure_package("ddgs")
ensure_package("openpyxl")
ensure_package("pandas")

import pandas as pd
from ddgs import DDGS

# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

RESULTS_PER_QUERY = 20
SLEEP_BETWEEN_SEARCHES = 1.0
MAX_BASE_QUERIES = None

OUTPUT_CSV = "relevamiento_estado_del_arte.csv"
OUTPUT_XLSX = "relevamiento_estado_del_arte.xlsx"

# ------------------------------------------------------------
# DESDUPLICACIÓN DE URLS
# ------------------------------------------------------------
DEDUPLICATE_URLS = True
URL_DEDUP_STRATEGY = "canonical"   # "exact" o "canonical"

# ------------------------------------------------------------
# FILTRADO DE DOMINIOS EXCLUIDOS
# ------------------------------------------------------------
EXCLUDED_DOMAINS = [
    "youtube.com",
    "youtu.be",
    "tiktok.com",
    "facebook.com",
    "instagram.com",
    "twitter.com",
    "x.com",
    "linkedin.com",
    "blogspot.com",
    "wordpress.com"
]

# ============================================================
# BUSCADORES LÓGICOS
# ============================================================

SEARCH_ENGINES = {
    "IA_MPF": {
        "campo": "IA_MPF",
        "required_sites": [
            # anteriores
            "mpf.gob.ar",
            "fiscalia.gov.co",
            "ministeriopublico.gob.pe",
            "minpublico.cl",
            "fiscal.es",
            "pgr.gob.mx",
            "fgr.org.mx",
            "saij.gob.ar",
            "poderjudicial.es",
            "scjn.gob.mx",

            # nuevos agregados
            "mpba.gov.ar",
            "fiscalia.gob.ec",
            "ministeriopublico.gob.bo",
            "ministeriopublico.gob.py",
            "fiscalia.gob.gt",
            "fiscalia.gob.hn",
            "mp.gob.sv",
            "ministeriopublico.gob.do",
            "fiscalia.gob.pa",
            "mp.gob.ve",
            "fiscalia.gob.ve",
            "fiscalia.go.cr",
            "ministeriopublico.gob.cr",
            "fiscalia.gub.uy",
            "ministeriopublico.gob.ni"
        ],
        "include_free_search": True,
    },
}

# ============================================================
# LISTAS DE KEYWORDS
# ============================================================

keywords_juridicas = [
    "ministerio público fiscal", "fiscalía", "investigación penal",
    "proceso penal", "procedimiento penal", "prueba digital", "evidencia digital",
    "cadena de custodia digital", "peritaje informático", "análisis forense digital"
]

keywords_informaticas = [
    "inteligencia artificial", "aprendizaje automático", "procesamiento de datos",
    "automatización de procesos", "análisis de datos", "minería de datos", "big data",
    "procesamiento de lenguaje natural", "reconocimiento de patrones", "análisis automatizado"
]

# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def normalize_space(text: str) -> str:
    if not text:
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()

def extract_domain(url: str) -> str:
    try:
        return urlparse(url).netloc.lower().replace("www.", "")
    except Exception:
        return ""

def is_excluded_domain(domain: str, excluded_domains: list[str]) -> bool:
    d = (domain or "").lower().replace("www.", "")
    for blocked in excluded_domains:
        b = (blocked or "").lower().replace("www.", "")
        if d == b or d.endswith("." + b):
            return True
    return False

def canonicalize_url(url: str) -> str:
    try:
        parsed = urlparse(normalize_space(url))
        scheme = (parsed.scheme or "https").lower()
        netloc = parsed.netloc.lower().replace("www.", "")
        path = parsed.path or "/"

        if path != "/":
            path = path.rstrip("/")
            if not path.startswith("/"):
                path = "/" + path

        raw_query = parsed.query.strip()
        query_pairs = []
        if raw_query:
            for part in raw_query.split("&"):
                if "=" in part:
                    k, v = part.split("=", 1)
                else:
                    k, v = part, ""
                k = k.strip()
                v = v.strip()

                if k.lower().startswith("utm_"):
                    continue
                if k.lower() in {"fbclid", "gclid", "igshid", "mc_cid", "mc_eid"}:
                    continue

                query_pairs.append((k, v))

        query_pairs = sorted(query_pairs, key=lambda x: (x[0], x[1]))
        normalized_query = "&".join(
            [f"{k}={v}" if v != "" else k for k, v in query_pairs]
        )

        return urlunparse((scheme, netloc, path, "", normalized_query, ""))
    except Exception:
        return normalize_space(url)

def get_dedup_key(url: str) -> str:
    if URL_DEDUP_STRATEGY == "canonical":
        return canonicalize_url(url)
    return normalize_space(url)

def classify_actor_type(domain: str, text: str) -> str:
    d = (domain or "").lower()
    t = (text or "").lower()

    if any(x in d for x in [".gov", ".gob", "fiscal", "judiciary", "justice", "poderjudicial", "boe.es"]):
        return "gobierno/justicia"

    if any(x in d for x in [".edu", ".ac.", "universidad", "university"]) or \
       any(x in t for x in ["universidad", "university", "facultad", "faculty", "research group", "grupo de investigación"]):
        return "academia"

    if any(x in t for x in [
        "editorial", "librería", "libreria", "revista", "publicación", "publicacion",
        "artículo", "articulo", "libro",
        "journal", "publisher", "publishing", "publication",
        "article", "paper", "book", "magazine"
    ]) or any(x in d for x in [
        "revista", "journal", "publisher", "books", "press"
    ]):
        return "editor"

    if any(x in d for x in [".org", ".int"]) or \
       any(x in t for x in ["ngo", "ong", "nonprofit", "asociación", "fundación", "foundation"]):
        return "ONG"

    if any(x in t for x in ["software", "platform", "plataforma", "startup", "company", "empresa", "provider"]) or \
       any(x in d for x in [".com", ".io", ".ai"]):
        return "empresa"

    return "otro"

def identify_country(domain: str) -> str:
    d = (domain or "").lower().replace("www.", "")

    # --- 1. MAPEO DIRECTO (PRIORIDAD ALTA) ---
    country_map = {
        "mpf.gob.ar": "Argentina",
        "mpba.gov.ar": "Argentina",
        "saij.gob.ar": "Argentina",
        "fiscalia.gov.co": "Colombia",
        "ministeriopublico.gob.pe": "Perú",
        "minpublico.cl": "Chile",
        "fiscal.es": "España",
        "poderjudicial.es": "España",
        "pgr.gob.mx": "México",
        "fgr.org.mx": "México",
        "scjn.gob.mx": "México",
        "fiscalia.gob.ec": "Ecuador",
        "ministeriopublico.gob.bo": "Bolivia",
        "ministeriopublico.gob.py": "Paraguay",
        "fiscalia.gob.gt": "Guatemala",
        "fiscalia.gob.hn": "Honduras",
        "mp.gob.sv": "El Salvador",
        "ministeriopublico.gob.do": "República Dominicana",
        "fiscalia.gob.pa": "Panamá",
        "mp.gob.ve": "Venezuela",
        "fiscalia.gob.ve": "Venezuela",
        "fiscalia.go.cr": "Costa Rica",
        "ministeriopublico.gob.cr": "Costa Rica",
        "fiscalia.gub.uy": "Uruguay",
        "ministeriopublico.gob.ni": "Nicaragua",
    }

    if d in country_map:
        return country_map[d]

    for known_domain, country in country_map.items():
        if d.endswith("." + known_domain):
            return country

    # --- 2. INFERENCIA POR SUFIJO DE PAÍS (ccTLD) ---
    tld_map = {
        ".ar": "Argentina",
        ".mx": "México",
        ".co": "Colombia",
        ".pe": "Perú",
        ".cl": "Chile",
        ".ec": "Ecuador",
        ".bo": "Bolivia",
        ".py": "Paraguay",
        ".uy": "Uruguay",
        ".ve": "Venezuela",
        ".pa": "Panamá",
        ".cr": "Costa Rica",
        ".gt": "Guatemala",
        ".hn": "Honduras",
        ".sv": "El Salvador",
        ".ni": "Nicaragua",
        ".do": "República Dominicana",
        ".es": "España",
    }

    for tld, country in tld_map.items():
        if d.endswith(tld):
            return country

    # --- 3. FALLBACK ---
    return "Sin identificar"

def build_base_queries_cartesian(
    keywords_juridicas: list[str],
    keywords_informaticas: list[str],
    max_queries: int | None = None
) -> list[str]:
    queries = []
    for kw_jur in keywords_juridicas:
        for kw_info in keywords_informaticas:
            queries.append(normalize_space(f"{kw_info} {kw_jur}"))
    queries = sorted(set(queries))
    if max_queries is not None:
        queries = queries[:max_queries]
    return queries

def compose_engine_queries(
    required_sites: list[str],
    base_query: str,
    include_free_search: bool = False
) -> list[str]:

    import unicodedata

    def remove_accents(text: str) -> str:
        return "".join(
            c for c in unicodedata.normalize("NFD", text)
            if unicodedata.category(c) != "Mn"
        )

    normalized = remove_accents(base_query)

    queries = []

    # --- BÚSQUEDA LIBRE ---
    if include_free_search:
        queries.append(base_query)

        if normalized != base_query:
            queries.append(normalized)

    # --- BÚSQUEDA EN REPOSITORIOS ---
    for site in required_sites:
        queries.append(f"{base_query} site:{site}")

    return sorted(set(queries))

def format_progress(current: int, total: int) -> str:
    if total <= 0:
        return "0.00%"
    return f"{(current / total) * 100:.2f}%"

# ============================================================
# BÚSQUEDA DDGS
# ============================================================

def search_ddgs(query: str, max_results: int = 10) -> list[dict]:
    results = []
    with DDGS() as ddgs:
        raw = ddgs.text(query, max_results=max_results)
        for item in raw:
            title = normalize_space(item.get("title", ""))
            link = normalize_space(item.get("href", ""))
            snippet = normalize_space(item.get("body", ""))
            if title and link:
                results.append({
                    "title": title,
                    "link": link,
                    "snippet": snippet,
                })
    return results

def normalize_search_result(field_name: str, query: str, item: dict) -> dict:
    url = normalize_space(item.get("link", ""))
    title = normalize_space(item.get("title", ""))
    snippet = normalize_space(item.get("snippet", ""))
    domain = extract_domain(url)
    country = identify_country(domain)

    page_title = title
    combined_text = " ".join([title, snippet, page_title])
    actor_type = classify_actor_type(domain, combined_text)

    return {
        "CAMPO": field_name,
        "PAIS": country,
        "DOMINIO": domain,
        "TITULO": title,
        "RESUMEN": snippet,
        "URL": url,
        "ACTOR": actor_type,
        "URL_NORMALIZADA": get_dedup_key(url),
        "CONSULTA": query,
    }

# ============================================================
# TEST INICIAL
# ============================================================

def test_search():
    test_query = "derecho penal inteligencia artificial site:fiscal.es"
    print("=== TEST INICIAL DDGS ===")
    print("Query:", test_query)
    results = search_ddgs(test_query, max_results=3)
    print("Resultados de prueba:", len(results))
    if results:
        print("Primer título:", results[0]["title"])
        print("Primera URL:", results[0]["link"])
    print()

# ============================================================
# EJECUCIÓN PRINCIPAL
# ============================================================

def main():
    test_search()

    base_queries = build_base_queries_cartesian(
        keywords_juridicas=keywords_juridicas,
        keywords_informaticas=keywords_informaticas,
        max_queries=MAX_BASE_QUERIES
    )

    print("=== RESUMEN ===")
    print("Modo de combinación:", "producto cartesiano absoluto")
    print("Base queries generadas:", len(base_queries))
    print(f"Desduplicación de URLs: {'sí' if DEDUPLICATE_URLS else 'no'}")
    print(f"Estrategia de desduplicación: {URL_DEDUP_STRATEGY}")
    print(f"Dominios excluidos: {', '.join(EXCLUDED_DOMAINS) if EXCLUDED_DOMAINS else '[ninguno]'}")
    print()

    rows = []

    for engine_name, engine_conf in SEARCH_ENGINES.items():
        campo = engine_conf["campo"]
        required_sites = engine_conf.get("required_sites", [])
        include_free_search = engine_conf.get("include_free_search", False)

        print(f"=== Buscador lógico: {engine_name} ===")
        print(f"Campo: {campo}")
        print(f"Búsqueda libre: {'sí' if include_free_search else 'no'}")
        print(f"Repositorios forzados: {', '.join(required_sites) if required_sites else '[ninguno]'}")
        print()

        engine_queries = []
        for base_query in base_queries:
            engine_queries.extend(
                compose_engine_queries(
                    required_sites=required_sites,
                    base_query=base_query,
                    include_free_search=include_free_search
                )
            )

        engine_queries = sorted(set(engine_queries))
        total_queries = len(engine_queries)

        print(f"Queries efectivas en este buscador: {total_queries}")
        print()

        for idx, final_query in enumerate(engine_queries, start=1):
            progress_pct = format_progress(idx, total_queries)
            print(f"[{idx}/{total_queries}] ({progress_pct}) {final_query}")

            try:
                results = search_ddgs(final_query, max_results=RESULTS_PER_QUERY)

                for item in results:
                    row = normalize_search_result(campo, final_query, item)
                    if row["URL"] and not is_excluded_domain(row["DOMINIO"], EXCLUDED_DOMAINS):
                        rows.append(row)

            except Exception as e:
                print(f"Error en '{final_query}' ({engine_name}): {e}")

            time.sleep(SLEEP_BETWEEN_SEARCHES)

        print()

    df = pd.DataFrame(rows)

    if df.empty:
        print("No se recuperaron resultados.")
        return

    total_before_dedup = len(df)

    if DEDUPLICATE_URLS:
        df = df.drop_duplicates(subset=["URL_NORMALIZADA"], keep="first").reset_index(drop=True)

    total_after_dedup = len(df)

    ordered_cols = [
        "CAMPO",
        "PAIS",
        "DOMINIO",
        "TITULO",
        "RESUMEN",
        "URL",
        "ACTOR",
        "CONSULTA",
    ]

    df = df.sort_values(
        by=["CAMPO", "PAIS", "CONSULTA", "ACTOR", "DOMINIO", "TITULO"],
        ascending=[True, True, True, True, True, True],
    ).reset_index(drop=True)

    print("=== RESUMEN DE DUPLICADOS ===")
    print("Filas antes de desduplicar:", total_before_dedup)
    print("Filas después de desduplicar:", total_after_dedup)
    print("Duplicados eliminados:", total_before_dedup - total_after_dedup)
    print()

    print("=== VISTA PREVIA ===")
    print(df[ordered_cols].head(15).to_string(index=False))

    df[ordered_cols].to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        df[ordered_cols].to_excel(writer, sheet_name="resultados", index=False)

    print()
    print("Archivos guardados:")
    print("-", OUTPUT_CSV)
    print("-", OUTPUT_XLSX)

    try:
        from google.colab import files
        files.download(OUTPUT_CSV)
        files.download(OUTPUT_XLSX)
    except Exception:
        pass

main()

=== TEST INICIAL DDGS ===
Query: derecho penal inteligencia artificial site:fiscal.es
Resultados de prueba: 3
Primer título: La inteligencia artificial y su impacto en la Administración de Justicia, a debate en un nuevo curso formativo de la Fiscalía General - fiscal.es
Primera URL: https://www.fiscal.es/-/la-inteligencia-artificial-y-su-impacto-en-la-administración-de-justicia-a-debate-en-un-nuevo-curso-formativo-de-la-fiscalía-general

=== RESUMEN ===
Modo de combinación: producto cartesiano absoluto
Base queries generadas: 100
Desduplicación de URLs: sí
Estrategia de desduplicación: canonical
Dominios excluidos: youtube.com, youtu.be, tiktok.com, facebook.com, instagram.com, twitter.com, x.com, linkedin.com, blogspot.com, wordpress.com

=== Buscador lógico: IA_MPF ===
Campo: IA_MPF
Búsqueda libre: sí
Repositorios forzados: mpf.gob.ar, fiscalia.gov.co, ministeriopublico.gob.pe, minpublico.cl, fiscal.es, pgr.gob.mx, fgr.org.mx, saij.gob.ar, poderjudicial.es, scjn.gob.mx, mpba.gov.ar, 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>